In [ ]:
import datetime

from funcs import \
    read_csv, \
    fill_missing_category_name, \
    check_missing_product_dimensions, \
    handle_missing_product_dimensions, \
    category_patternization, \
    clean_text_columns, \
    convert_numeric_columns, \
    convert_date_to_br_fmt

## Ingestão de dados

In [ ]:
relative_path = './data/'

In [ ]:
orders = read_csv(relative_path + 'olist_orders_dataset.csv')
products = read_csv(relative_path + 'olist_products_dataset.csv')

### Verficando as colunas presentes

In [ ]:
products[0]

In [ ]:
orders[0]

In [ ]:
# Numero total de registros no dataset
total_initial_products = len(products)

## 1. Validação e tratamento de dados ausentes

- Substituir categorias faltantes por `sem categoria`

- Verificar produtos com dimensões físicas faltantes e definir uma regra de corte para eles

In [ ]:
dimensions_columns = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
    ]

In [ ]:
print(f"Total de produtos no dataset: {total_initial_products}")

check_missing_product_dimensions(products, dimensions_columns)

Como são poucos produtos com dimensões físicas faltantes em relação ao tamanho total do dataset, foi decidido remover essas entradas.

In [ ]:
missing_category_count = fill_missing_category_name(products)
missing_dimensions_count = handle_missing_product_dimensions(products, dimensions_columns)

## 2. Padronização de Strings e Regex

- Garantir que nome de categorias sejam convertidos para lower

- Remoção de espaços em branco das extremidades das strings

- Usar expressões regulares para remover caracteres especiais

- Converter valores numéricos para float

### Identificando os tipos dos dados

In [ ]:
products[0]

In [ ]:
orders[0]

In [ ]:
text_columns_products = ['product_id', 'product_category_name']

numeric_columns_products = [
    'product_name_lenght',
    'product_description_lenght',
    'product_photos_qty',
    'product_weight_g',
    'product_length_cm',
    'product_height_cm',
    'product_width_cm'
]

text_columns_orders = ['order_status']

### Aplicando as mudanças

In [ ]:
category_patternization(products)
convert_numeric_columns(products, numeric_columns_products)
clean_text_columns(products, text_columns_products)

clean_text_columns(orders, text_columns_orders)

## 3. Regra de negócio

- Investigar relacão entre `order_delivered_customer_date` `order_status`

In [ ]:
# Status possíveis para os pedidos
status_list  = set([o['order_status'] for o in orders])

# Contagem de numero pedidos por status:
status_count_dict = { status : 0 for status in status_list}

# Percorrer o dataset para contar os status
for o in orders:
    status_count_dict[o['order_status']] = status_count_dict[o['order_status']] + 1

status_count_dict

In [ ]:
# Contagem de status dos pedidos com data de entrega vazia:

empty_date_status_count_dict = { status : 0 for status in status_list}

for o in orders:
    if (o["order_delivered_customer_date"] == '' or o["order_delivered_customer_date"] is None):
        empty_date_status_count_dict[o['order_status']] = empty_date_status_count_dict[o['order_status']] + 1

empty_date_status_count_dict

In [ ]:
empty_date_status_percentages = { 
                                status : 
                                    empty_date_status_count_dict[status] / sum(empty_date_status_count_dict.values()) * 100
                                for status in status_list 
                                }

empty_date_status_percentages

## 4. Formatação temporal

- Converter a coluna `order_approved_at` para formato brasileiro

### Verificando como é a formatação atual

In [ ]:
orders[0]['order_approved_at']

### Realizando a conversão

In [ ]:
colunas_para_converter = ['order_approved_at']

In [ ]:
convert_date_to_br_fmt(orders, colunas_para_converter)

Verificando o resultado

In [ ]:
orders[0]['order_approved_at']

# 5 Relatório Simplificado

- Realizar uma análise dos resultados

### Número total de linhas em cada dataset

In [ ]:
print(f"Número total de produtos inicial: {total_initial_products}")
print(f"Número total de produtos final: {len(products)}")
print(f"Numero total de pedidos: {len(orders)}")

### Produtos ajustados ou removidos

In [ ]:
print("--------------------------------------------------------------------")
print("\t\tProdutos ajustados ou removidos")
print("--------------------------------------------------------------------")

print (f"Produtos com categorias faltantes ajustados: {missing_category_count} ({missing_category_count/total_initial_products:.6f}%)")
print (f"Produtos com dimensões faltantes deletados: {missing_dimensions_count} ({missing_dimensions_count/total_initial_products:.6f}%)")

### Número de pedidos por status

In [ ]:
print("--------------------------------------------------------------------")
print("\t\tNúmero total de pedidos por status:")
print("--------------------------------------------------------------------")
for k, v in status_count_dict.items():
    print(f"{k.capitalize()} : {v}")

### Pedidos sem data de entrega

Verificando a hipótese da Olist: pedidos com data de entrega vazia são **obrigatoriamente** cancelados?

In [ ]:
print("--------------------------------------------------------------------")
print("Número de pedidos sem data de entrega categorizados por status:")
print("--------------------------------------------------------------------")
for k, v in   empty_date_status_count_dict.items():
    print(f"{k.capitalize()} : {v}")


print("\n--------------------------------------------------------------------")
print("Porcentagem de pedidos sem data de entrega categorizados por status:")
print("--------------------------------------------------------------------")
for k, v in empty_date_status_percentages.items():
    print(f"{k.capitalize()} : {v:.1f}%")

Apesar de os pedidos `cancelled` (cancelados) serem uma parcela relevante (**20,9%**) dos pedidos com datas de entrega vazias (`order_delivered_customer_date`), o `status` com maior participação percentual nessa categoria é `shipped` (despachados) (**37,3%**).

Portanto, a hipótese inicial de que apenas pedidos cancelados possuem data de entrega vazia é **refutada**.

### Sumário estatístico resumido

In [ ]:
print(f"Número total de produtos inicial: {total_initial_products}")
print(f"Número total de produtos final: {len(products)}")
print(f"Numero total de pedidos: {len(orders)}")

print("--------------------------------------------------------------------")

print (f"Produtos com categorias faltantes ajustados: {missing_category_count} ({missing_category_count/total_initial_products:.6f}% do total inicial)")
print (f"Produtos com dimensões faltantes deletados: {missing_dimensions_count} ({missing_dimensions_count/total_initial_products:.6f}% do total inicial)")

print("--------------------------------------------------------------------")

print(f"Total de pedidos cancelados identificados: {empty_date_status_count_dict['canceled']}")